In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName("insurance ETL pipeline").getOrCreate()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/15 22:50:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
print(spark.version)

4.2.0


In [6]:
from pyspark.sql.types import StructType , StructField , StringType , FloatType , DateType

In [18]:
policies_schema = StructType([
	StructField("policy_id" , StringType()),
	StructField("client_id" , StringType()),
	StructField("country" , StringType()),
    StructField("product" , StringType()),
    StructField("premium_amount" , FloatType()),
    StructField("currrency" , FloatType()),
    StructField("start_date" , DateType()),
    StructField("status" , StringType()),
])

claims_schema = StructType([
	StructField("claim_id" , StringType()),
	StructField("policy_id" , StringType()),
	StructField("claim_date" , DateType()),
 	StructField("claim_type" , StringType()),
	StructField("claim_amount" , FloatType()),
	StructField("status" , StringType())
])

In [19]:
policies_df = spark.read.csv("./../data/policies.csv" , schema = policies_schema )
claims_df = spark.read.csv("./../data/claims.csv" , schema = claims_schema )

In [20]:
print("Policies DataFrame:")
policies_df.show()
print("Claims DataFrame:")
claims_df.show()

Policies DataFrame:
+---------+---------+-------+----------+--------------+---------+----------+---------+
|policy_id|client_id|country|   product|premium_amount|currrency|start_date|   status|
+---------+---------+-------+----------+--------------+---------+----------+---------+
|policy_id|client_id|country|   product|          NULL|     NULL|      NULL|   status|
| POL00001|  CLI0027|     FR|      Auto|       1826.24|     NULL|2025-10-17|Suspendue|
| POL00002|  CLI0007|     LU|      Auto|        246.15|     NULL|2025-02-23|Suspendue|
| POL00003|  CLI0041|     LU|Habitation|       1177.24|     NULL|      NULL| Résiliée|
| POL00004|  CLI0089|     BE|     Santé|        406.91|     NULL|      NULL|   Active|
| POL00005|  CLI0093|     FR|Prévoyance|        354.94|     NULL|2025-03-28|Suspendue|
| POL00006|  CLI0098|     BE|     Santé|       2367.19|     NULL|      NULL|Suspendue|
| POL00007|  CLI0019|     BE|     Santé|         968.7|     NULL|2025-06-01|   Active|
| POL00008|  CLI0143|  

In [21]:
print(f"Total number of policies: {policies_df.count()}")
print(f"Total number of claims: {claims_df.count()}")


Total number of policies: 226
Total number of claims: 487


In [25]:
# Identifier et quantifier les valeurs manquantes par colonne, pour les deux DataFrames.

def count_missing_values(df):
	missing_counts = {}
	for col in df.columns:
		missing_count = df.filter(df[col].isNull()).count()
		missing_counts[col] = missing_count
	return missing_counts



In [24]:
print("Missing values in policies DataFrame:")
policies_missing_counts = count_missing_values(policies_df)

for col, count in policies_missing_counts.items():
	print(f"Missing values in column '{col}': {count}")

Missing values in policies DataFrame:
Missing values in column 'policy_id': 0
Missing values in column 'client_id': 0
Missing values in column 'country': 4
Missing values in column 'product': 0
Missing values in column 'premium_amount': 10
Missing values in column 'currrency': 226
Missing values in column 'start_date': 74
Missing values in column 'status': 0


In [26]:
print("Missing values in claims DataFrame:")
claims_missing_counts = count_missing_values(claims_df)

for col, count in claims_missing_counts.items():
	print(f"Missing values in column '{col}': {count}")

Missing values in claims DataFrame:
Missing values in column 'claim_id': 0
Missing values in column 'policy_id': 0
Missing values in column 'claim_date': 167
Missing values in column 'claim_type': 0
Missing values in column 'claim_amount': 11
Missing values in column 'status': 0


In [ ]:
# Afficher les doublons exacts dans policies et claims
# 3. Identifier les doublons exacts dans policies et les supprimer proprement

print("Exact duplicates in policies DataFrame:")
exact_duplicates_policies = policies_df.groupBy(policies_df.columns).count().filter("count > 1")
exact_duplicates_policies.show()



Exact duplicates in policies DataFrame:
+---------+---------+-------+----------+--------------+---------+----------+---------+-----+
|policy_id|client_id|country|   product|premium_amount|currrency|start_date|   status|count|
+---------+---------+-------+----------+--------------+---------+----------+---------+-----+
| POL00138|  CLI0084|     LU|     Santé|          NULL|     NULL|2022-12-18| Résiliée|    2|
| POL00043|  CLI0148|     CH|Prévoyance|       1525.78|     NULL|      NULL|   Active|    2|
| POL00084|  CLI0114|     CH|      Auto|        1874.1|     NULL|2023-07-24|Suspendue|    2|
| POL00219|  CLI0056|     CH|Prévoyance|       1738.31|     NULL|2026-04-10|Suspendue|    2|
| POL00039|  CLI0064|     FR|Habitation|       1815.17|     NULL|2023-07-01|Suspendue|    2|
+---------+---------+-------+----------+--------------+---------+----------+---------+-----+



In [ ]:

# select * from policies where policy_id in ( select policy_id from policies group by policy_id having count(*) > 1 )

policies_df.filter(policies_df["policy_id"].isin([row["policy_id"] for row in exact_duplicates_policies.select("policy_id").collect()])).orderBy("policy_id").show()

+---------+---------+-------+----------+--------------+---------+----------+---------+
|policy_id|client_id|country|   product|premium_amount|currrency|start_date|   status|
+---------+---------+-------+----------+--------------+---------+----------+---------+
| POL00039|  CLI0064|     FR|Habitation|       1815.17|     NULL|2023-07-01|Suspendue|
| POL00039|  CLI0064|     FR|Habitation|       1815.17|     NULL|2023-07-01|Suspendue|
| POL00043|  CLI0148|     CH|Prévoyance|       1525.78|     NULL|      NULL|   Active|
| POL00043|  CLI0148|     CH|Prévoyance|       1525.78|     NULL|      NULL|   Active|
| POL00084|  CLI0114|     CH|      Auto|        1874.1|     NULL|2023-07-24|Suspendue|
| POL00084|  CLI0114|     CH|      Auto|        1874.1|     NULL|2023-07-24|Suspendue|
| POL00138|  CLI0084|     LU|     Santé|          NULL|     NULL|2022-12-18| Résiliée|
| POL00138|  CLI0084|     LU|     Santé|          NULL|     NULL|2022-12-18| Résiliée|
| POL00219|  CLI0056|     CH|Prévoyance|   

In [34]:
policies_df = policies_df.dropDuplicates()

print("Policies DataFrame after removing exact duplicates:")
policies_df.show()

Policies DataFrame after removing exact duplicates:
+---------+---------+-------+----------+--------------+---------+----------+---------+
|policy_id|client_id|country|   product|premium_amount|currrency|start_date|   status|
+---------+---------+-------+----------+--------------+---------+----------+---------+
| POL00155|  CLI0050|     BE|Habitation|        867.02|     NULL|2024-01-24|Suspendue|
| POL00186|  CLI0012|     CH|      Auto|          NULL|     NULL|2025-08-06| Résiliée|
| POL00099|  CLI0012|     FR|      Auto|       1164.97|     NULL|2023-10-27| Résiliée|
| POL00141|  CLI0061|     CH|     Santé|       1794.78|     NULL|2022-07-30| Résiliée|
| POL00166|  CLI0045|     FR|     Santé|        476.17|     NULL|2026-05-15| Résiliée|
| POL00137|  CLI0020|     FR|Prévoyance|      -2093.12|     NULL|2023-10-04| Résiliée|
| POL00180|  CLI0147|     BE|Habitation|        705.55|     NULL|      NULL|Suspendue|
| POL00125|  CLI0025|     BE|Prévoyance|        932.43|     NULL|      NULL| R